# Part 4 — Deep Q-Network (DQN) Training
**Optimization 2 · Project 3 · Connect-4**

Run on a **GPU runtime** in Colab:  
`Runtime → Change runtime type → T4 GPU`

### What this notebook does
- Builds a DQN with a **tanh** output layer (as specified in assignment)
- Uses **epsilon-greedy** exploration during training
- Maintains a **replay buffer** and a **target network**
- Trains against: Random, StrongRule, Josh CNN, Transformer, PG model, and past DQN snapshots
- Saves checkpoints to Google Drive so crashes don't wipe progress
- Generates win-rate, loss, and epsilon plots for the report

### Files to upload in Step 2
| File | Where to get it |
|------|-----------------|
| `connect4_env.py` | repo root |
| `loader.py` | `models/loader.py` |
| `josh_cnn.h5` | `models/josh_cnn.h5` |
| `connect4_transformer__1_.pt` | `models/` |
| `m1_pg_final.keras` | `part1_pg/best_model/` |

In [ ]:
# ── Step 1: Check GPU ──────────────────────────────────────────────────────
import tensorflow as tf
print('GPUs available:', tf.config.list_physical_devices('GPU'))
print('TF version:', tf.__version__)

In [ ]:
# ── Step 2: Upload required files ─────────────────────────────────────────
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── Step 3: Wire up folders ────────────────────────────────────────────────
import os, shutil

os.makedirs('models',                  exist_ok=True)
os.makedirs('part4_dqn/checkpoints',   exist_ok=True)
os.makedirs('part4_dqn/best_model',    exist_ok=True)
os.makedirs('part1_pg/best_model',     exist_ok=True)

for fname, dest in [
    ('loader.py',                    'models/loader.py'),
    ('josh_cnn.h5',                  'models/josh_cnn.h5'),
    ('connect4_transformer__1_.pt',  'models/connect4_transformer__1_.pt'),
    ('m1_pg_final.keras',            'part1_pg/best_model/m1_pg_final.keras'),
]:
    if os.path.exists(fname):
        shutil.copy(fname, dest)
        print(f'  {fname} -> {dest}')

with open('models/__init__.py', 'w') as f:
    f.write('')

print('models/:', os.listdir('models'))

In [ ]:
# ── Step 4: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Connect4_DQN_Checkpoints/'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive checkpoints -> {DRIVE_DIR}')
print('Existing:', os.listdir(DRIVE_DIR))

In [ ]:
# ── Step 5: Hyperparameters ────────────────────────────────────────────────
GAMMA               = 0.99
LR                  = 1e-4
BATCH_SIZE          = 64        # keep fixed — TF retraces if this changes
REPLAY_BUFFER_SIZE  = 50000
TARGET_UPDATE_EVERY = 500       # env steps between target network syncs
MIN_BUFFER_SIZE     = 1000      # wait until buffer has this many samples
EPSILON_START       = 1.0
EPSILON_END         = 0.05
EPSILON_DECAY_STEPS = 50000
N_EPISODES          = 3000
RANDOM_INIT_MOVES   = 3
TRAIN_EVERY         = 4
ADD_TO_POOL_EVERY   = 200
MAX_OWN_SNAPSHOTS   = 6
EVAL_EVERY          = 200
EVAL_GAMES          = 100
CHECKPOINT_DIR      = 'part4_dqn/checkpoints/'
BEST_MODEL_DIR      = 'part4_dqn/best_model/'
print('Hyperparameters set.')

In [ ]:
# ── Step 6: Imports ────────────────────────────────────────────────────────
import random, collections
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch
import torch.nn as nn

from connect4_env import (
    encode_board, legal_moves, game_over, make_board, step,
    ModelAgent, RandomAgent, StrongRuleAgent, evaluate_agents,
    find_winning_move
)
from models.loader import load_agent

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('Imports OK')

In [ ]:
# ── Step 7: Transformer opponent ───────────────────────────────────────────
# The Transformer is PyTorch so we define the class here to load it,
# then wrap it in an Agent that follows the same interface as ModelAgent.

class Connect4Transformer(nn.Module):
    def __init__(self, num_patches=42, patch_size=2, hidden_dim=128,
                 num_layers=6, num_heads=8, mlp_dim=256,
                 dropout_rate=0.3, num_classes=7):
        super().__init__()
        self.patch_embedding = nn.Linear(patch_size, hidden_dim)
        self.row_embedding   = nn.Embedding(6, hidden_dim // 2)
        self.col_embedding   = nn.Embedding(7, hidden_dim // 2)
        self.class_token     = nn.Parameter(torch.randn(1, 1, hidden_dim))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=mlp_dim,
            dropout=dropout_rate, activation='gelu',
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embedding(x)
        ri = torch.arange(6, device=x.device).repeat_interleave(7)
        ci = torch.arange(7, device=x.device).repeat(6)
        pos = torch.cat([self.row_embedding(ri), self.col_embedding(ci)], dim=-1)
        x = x + pos.unsqueeze(0)
        x = torch.cat([self.class_token.expand(B, -1, -1), x], dim=1)
        x = self.transformer(x)
        return self.head(self.norm(x[:, 0]))


class TransformerAgent:
    """PyTorch Transformer wrapped to match Agent interface."""
    def __init__(self, model, strong=True):
        self.model = model.eval()
        self.strong = strong

    def select_move(self, board, player):
        if self.strong:
            col = find_winning_move(board, player)
            if col is not None: return col
            col = find_winning_move(board, -player)
            if col is not None: return col
        legal = legal_moves(board)
        enc   = np.zeros((6, 7, 2), dtype=np.float32)
        enc[:, :, 0] = (board == player)
        enc[:, :, 1] = (board == -player)
        patches = torch.tensor(enc.reshape(1, 42, 2))
        with torch.no_grad():
            logits = self.model(patches)[0].numpy()
        mask = np.full(7, -1e9); mask[legal] = 0.0
        return int(np.argmax(logits + mask))


# Load transformer weights
transformer_model = Connect4Transformer()
sd = torch.load('models/connect4_transformer__1_.pt', map_location='cpu', weights_only=False)
if not isinstance(sd, dict):
    sd = sd.state_dict()
transformer_model.load_state_dict(sd)
transformer_agent = TransformerAgent(transformer_model)
print('Transformer loaded OK.')

In [ ]:
# ── Step 8: Build DQN ─────────────────────────────────────────────────────
# Architecture matches PG network but final activation is tanh not softmax
# Input:  (batch, 6, 7, 2)
# Output: (batch, 7)  Q-values in [-1, 1]

def build_dqn():
    inp = keras.Input(shape=(6, 7, 2))
    x   = layers.Conv2D(64,  3, padding='same', activation='relu')(inp)
    x   = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x   = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(7, activation='tanh', name='q_values')(x)  # tanh as specified
    return keras.Model(inp, out, name='Connect4DQN')

dqn_online = build_dqn()
dqn_target = build_dqn()
dqn_target.set_weights(dqn_online.get_weights())
optimizer  = keras.optimizers.Adam(learning_rate=LR)
dqn_online.summary()

In [ ]:
# ── Step 9: Replay Buffer ──────────────────────────────────────────────────

class ReplayBuffer:
    def __init__(self, maxlen=REPLAY_BUFFER_SIZE):
        self.buf = collections.deque(maxlen=maxlen)

    def add(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, n=BATCH_SIZE):
        batch = random.sample(self.buf, n)
        s, a, r, s2, d = zip(*batch)
        return (np.array(s,  dtype=np.float32),
                np.array(a,  dtype=np.int32),
                np.array(r,  dtype=np.float32),
                np.array(s2, dtype=np.float32),
                np.array(d,  dtype=np.float32))

    def __len__(self): return len(self.buf)

replay_buffer = ReplayBuffer()
print('Replay buffer ready.')

In [ ]:
# ── Step 10: Opponent pool ─────────────────────────────────────────────────
# First FIXED_POOL_SIZE entries are NEVER removed.
# DQN snapshots are added every ADD_TO_POOL_EVERY episodes.

def _keras_agent(model):
    return ModelAgent(model, sample=True, strong=True)

josh_agent = load_agent('josh_cnn')
pg_model   = keras.models.load_model('part1_pg/best_model/m1_pg_final.keras', compile=False)
pg_agent   = _keras_agent(pg_model)

FIXED_POOL_SIZE = 5
opponent_pool   = [
    RandomAgent(),        # 0 - never removed
    StrongRuleAgent(),    # 1 - never removed
    josh_agent,           # 2 - never removed (original MCTS CNN)
    transformer_agent,    # 3 - never removed
    pg_agent,             # 4 - never removed
]
own_snapshot_count = 0
print(f'Pool size: {len(opponent_pool)}')

In [ ]:
# ── Step 11: Helper functions ──────────────────────────────────────────────

total_steps = 0

def get_epsilon():
    frac = min(total_steps / EPSILON_DECAY_STEPS, 1.0)
    return EPSILON_START + frac * (EPSILON_END - EPSILON_START)

def select_move_dqn(board, player, epsilon):
    legal = legal_moves(board)
    if random.random() < epsilon:
        return random.choice(legal)
    enc   = encode_board(board, player)[np.newaxis].astype(np.float32)
    qvals = dqn_online(enc, training=False).numpy()[0]
    mask  = np.full(7, -1e9); mask[legal] = 0.0
    return int(np.argmax(qvals + mask))

@tf.function
def train_step(states, actions, rewards, next_states, dones):
    """Bellman update: target = r + gamma * max Q_target(s') * (1-done)"""
    max_next = tf.reduce_max(dqn_target(next_states, training=False), axis=1)
    targets  = rewards + GAMMA * max_next * (1.0 - dones)
    with tf.GradientTape() as tape:
        qvals    = dqn_online(states, training=True)
        indices  = tf.stack([tf.range(BATCH_SIZE, dtype=tf.int32), actions], axis=1)
        chosen_q = tf.gather_nd(qvals, indices)
        loss     = tf.reduce_mean(tf.square(targets - chosen_q))
    grads, _ = tf.clip_by_global_norm(
        tape.gradient(loss, dqn_online.trainable_variables), 1.0)
    optimizer.apply_gradients(zip(grads, dqn_online.trainable_variables))
    return loss

def sync_target():
    dqn_target.set_weights(dqn_online.get_weights())

def play_episode(m2_agent, epsilon):
    global total_steps
    dqn_player = random.choice([1, -1])
    board = make_board(); current = 1

    # Random warm-start (not recorded)
    for _ in range(RANDOM_INIT_MOVES):
        cols = legal_moves(board)
        if not cols: break
        board, _ = step(board, np.random.choice(cols), current)
        done, winner = game_over(board)
        if done: return winner
        current = -current

    while True:
        cols = legal_moves(board)
        if not cols: return 0

        if current == dqn_player:
            state = encode_board(board, current).astype(np.float32)
            col   = select_move_dqn(board, current, epsilon)
            total_steps += 1
        else:
            col = m2_agent.select_move(board, current)

        board, _ = step(board, col, current)
        done, winner = game_over(board)

        if current == dqn_player:
            if done:
                reward = 1.0 if winner == dqn_player else (-1.0 if winner != 0 else 0.1)
            else:
                reward = 0.0
            next_state = encode_board(board, dqn_player).astype(np.float32)
            replay_buffer.add(state, col, reward, next_state, float(done))

        if done: return winner
        current = -current

print('Helpers defined.')

In [ ]:
# ── Step 12: Training loop ─────────────────────────────────────────────────
log_loss, log_wr_strong, log_wr_pg = [], [], []
log_wr_transformer, log_wr_random  = [], []
log_iters, log_epsilon             = [], []

eval_strong      = StrongRuleAgent()
eval_pg          = ModelAgent(pg_model,          sample=False, strong=True)
eval_transformer = TransformerAgent(transformer_model)
eval_random      = RandomAgent()

own_snapshot_count = 0
step_loss_accum    = []
best_win_rate      = 0.0

print(f'Training DQN for {N_EPISODES} episodes...\n')

for episode in range(1, N_EPISODES + 1):

    epsilon  = get_epsilon()
    m2_agent = random.choice(opponent_pool)
    play_episode(m2_agent, epsilon)

    # Train
    if len(replay_buffer) >= MIN_BUFFER_SIZE and total_steps % TRAIN_EVERY == 0:
        s, a, r, s2, d = replay_buffer.sample()
        loss = train_step(
            tf.constant(s), tf.constant(a), tf.constant(r),
            tf.constant(s2), tf.constant(d)
        )
        step_loss_accum.append(float(loss))

    # Sync target network
    if total_steps > 0 and total_steps % TARGET_UPDATE_EVERY == 0:
        sync_target()

    # Evaluate
    if episode % EVAL_EVERY == 0:
        dqn_eval = ModelAgent(dqn_online, sample=False, strong=True)
        res_s = evaluate_agents(dqn_eval, eval_strong,      n_games=EVAL_GAMES)
        res_p = evaluate_agents(dqn_eval, eval_pg,          n_games=EVAL_GAMES)
        res_t = evaluate_agents(dqn_eval, eval_transformer, n_games=EVAL_GAMES)
        res_r = evaluate_agents(dqn_eval, eval_random,      n_games=EVAL_GAMES)

        avg_loss = np.mean(step_loss_accum) if step_loss_accum else 0.0
        step_loss_accum = []

        log_loss.append(avg_loss)
        log_wr_strong.append(res_s['win_rate'])
        log_wr_pg.append(res_p['win_rate'])
        log_wr_transformer.append(res_t['win_rate'])
        log_wr_random.append(res_r['win_rate'])
        log_iters.append(episode)
        log_epsilon.append(epsilon)

        print(f'[{episode:>5}] eps={epsilon:.3f} loss={avg_loss:.4f} | '
              f'Strong:{res_s["win_rate"]:.0%} '
              f'PG:{res_p["win_rate"]:.0%} '
              f'Transformer:{res_t["win_rate"]:.0%} '
              f'Random:{res_r["win_rate"]:.0%} '
              f'buf={len(replay_buffer)}')

        # Track best model
        if res_s['win_rate'] > best_win_rate:
            best_win_rate = res_s['win_rate']
            dqn_online.save(BEST_MODEL_DIR + 'dqn_best.keras')
            print(f'  New best: {best_win_rate:.1%} vs Strong')

    # Add DQN snapshot to pool
    if episode % ADD_TO_POOL_EVERY == 0:
        snap = build_dqn()
        snap.set_weights(dqn_online.get_weights())
        opponent_pool.append(_keras_agent(snap))
        own_snapshot_count += 1
        if own_snapshot_count > MAX_OWN_SNAPSHOTS:
            opponent_pool.pop(FIXED_POOL_SIZE)  # remove oldest DQN snapshot
            own_snapshot_count -= 1
        ckpt = f'{CHECKPOINT_DIR}dqn_ep{episode}.keras'
        dqn_online.save(ckpt)
        try:
            shutil.copy(ckpt, DRIVE_DIR)
            print(f'  Checkpoint + Drive backup. Pool={len(opponent_pool)}')
        except Exception:
            print(f'  Checkpoint saved. Pool={len(opponent_pool)}')

print('\nTraining complete!')

In [ ]:
# ── Step 13: Save final model ──────────────────────────────────────────────
import shutil
final_path = BEST_MODEL_DIR + 'dqn_final.keras'
dqn_online.save(final_path)
print(f'Saved: {final_path}')
try:
    shutil.copy(final_path, DRIVE_DIR + 'dqn_final.keras')
    print('Backed up to Drive.')
except Exception as e:
    print(f'Drive backup skipped: {e}')

In [ ]:
# ── Step 14: Training curves ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('DQN Training — Connect-4', fontsize=13)

axes[0].plot(log_iters, log_loss, marker='o', color='steelblue')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')

axes[1].plot(log_iters, log_wr_strong,      marker='o', label='vs Strong')
axes[1].plot(log_iters, log_wr_pg,          marker='o', label='vs PG Model')
axes[1].plot(log_iters, log_wr_transformer, marker='o', label='vs Transformer')
axes[1].plot(log_iters, log_wr_random,      marker='o', label='vs Random')
axes[1].axhline(0.5, color='gray', linestyle='--', label='50% baseline')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Win Rate')
axes[1].set_title('Win Rates vs Opponents')
axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1)

axes[2].plot(log_iters, log_epsilon, marker='o', color='darkorange')
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Epsilon')
axes[2].set_title('Epsilon Decay')

plt.tight_layout()
plot_path = BEST_MODEL_DIR + 'dqn_training_curves.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved: {plot_path}')

In [ ]:
# ── Step 15: Download results ──────────────────────────────────────────────
from google.colab import files
files.download(BEST_MODEL_DIR + 'dqn_final.keras')
files.download(BEST_MODEL_DIR + 'dqn_best.keras')
files.download(BEST_MODEL_DIR + 'dqn_training_curves.png')